# zk-Crown · P2.7 · Trigger-to-clean ratio sweep

**Runtime > Change runtime type > T4 GPU before you start.**

This trains **five** behavioral-watermark models. Each one uses exactly the P2.3 recipe: same
45,000 clean CIFAR-10 images, seed 1337, clean batch order, 60 epochs and trigger bundle. The only
difference is how many trigger samples are mixed in:

| run | triggers per batch | on every k-th batch | trigger samples / epoch | trigger : clean |
|---|---|---|---|---|
| `p2.7_r0001` | 1 | 64 | 6 | 0.013% |
| `p2.7_r0005` | 1 | 16 | 22 | 0.049% |
| `p2.7_r0020` | 1 | 4 | 88 | 0.196% |
| `p2.7_r0078` | 1 | 1 | 352 | 0.782% |
| `p2.7_r1252` | 16 | 1 | 5,632 | 12.5% |

P2.3 (4 per batch, 3.13%) and the clean `W` (0%) are reused, not retrained.

**Run order:** top to bottom (Runtime > Run all is fine). Cell 2 is the only cell you may need to
edit. Every check raises on failure, so Run all stops at the first problem.

**Expected runtime:** roughly 1 h 45 min to 1 h 55 min on a T4 in total, about 20 minutes per run.
That is an *estimate* from P2.3's measured 1,182 s for one run of this recipe.

**Can be split across sessions.** Section 7 has one cell per run. A run that is already complete on
Drive is skipped, and a run cut off mid-way resumes from its last epoch. Stopping after any cell
and continuing later is safe.

**Secrets.** The bundle is read from Drive and never printed. The result JSONs contain only its
SHA-256 digest.

**This notebook is a thin wrapper** (CLAUDE.md section 3). All logic is in `src/`,
`experiments/p2_3_train_watermarked.py` and `experiments/p2_7_ratio_sweep.py`.

## 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Configuration

In [ ]:
# ---- The only cell you may need to edit ------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/zk-crown"

# Where you put P2.7_colab.zip. If this file is not found, the next cell offers
# a manual upload instead.
ZIP_PATH = f"{DRIVE_ROOT}/handoff/P2.7_colab.zip"

# The SECRET trigger bundle, the same file P2.3 used. Already on Drive if you kept it.
BUNDLE_PATH = f"{DRIVE_ROOT}/secrets/trigger_bundle.npz"
# Its digest. Every run refuses any other bundle. Do not edit.
EXPECTED_TRIGGER_SHA256 = "fbd65ec730a3555c5921f13d1a5d4840b6310261ddcaf31d3a724195122baec8"

WORK_DIR = "/content/zk-crown"
# Outside WORK_DIR, which is wiped on every unpack, so a re-run after a
# disconnect in the same runtime does not download CIFAR-10 again.
DATA_ROOT = "/content/data"
EPOCHS = 60              # same as P2.3; do not change
SEED = 1337              # same as P2.3; do not change
BATCH_SIZE = 128         # same as P2.3; do not change
MAX_MINUTES = 90         # per run; stop cleanly before Colab does; re-run to continue

## 3. What GPU did we get?

Section 2.1: a T4 is expected, not guaranteed. If this says "no GPU", fix the runtime type before
going further.

In [ ]:
!nvidia-smi

## 4. Unpack the handoff zip and find the bundle

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# If Drive is not mounted, /content/drive/MyDrive/... would be created on local
# disk and every checkpoint would silently vanish with the session.
if DRIVE_ROOT.startswith("/content/drive/") and not os.path.ismount("/content/drive"):
    raise RuntimeError("Drive is not mounted. Run section 1 first.")

# Fail now, not after the tests, if the secret bundle is missing.
if not Path(BUNDLE_PATH).is_file():
    raise FileNotFoundError(
        f"trigger bundle not found at {BUNDLE_PATH}. Upload secrets/trigger_bundle.npz "
        "from your local repo to that Drive path (see P2.7_INSTRUCTIONS.md)."
    )
print("trigger bundle found:", BUNDLE_PATH, f"({Path(BUNDLE_PATH).stat().st_size / 1024:.0f} KB)")

work = Path(WORK_DIR)
if work.exists():
    shutil.rmtree(work)          # always unpack fresh; Drive holds the state that matters
work.mkdir(parents=True)

zip_path = Path(ZIP_PATH)
if not zip_path.is_file():
    print(f"{zip_path} not found. Upload P2.7_colab.zip now:")
    from google.colab import files

    uploaded = files.upload()
    name = next(iter(uploaded))
    zip_path = Path("/content") / name

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(work)

if not (work / "src").is_dir():
    inner = [d for d in work.iterdir() if d.is_dir() and (d / "src").is_dir()]
    if not inner:
        raise RuntimeError(f"no src/ found under {work}")
    work = inner[0]

REPO_DIR = str(work)
os.chdir(REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print(sorted(p.name for p in Path(REPO_DIR).iterdir()))
build_info = Path(REPO_DIR, "BUILD_INFO.json")
print(build_info.read_text() if build_info.is_file() else "no BUILD_INFO.json in the zip")


def run(args):
    """Run a command, stream its output into the cell, raise if it fails.

    `!cmd` does neither reliably: a non-zero exit does not fail the cell, so
    Run all would carry on past a failed test suite into the real run.
    """
    print("$", *args, flush=True)
    print(flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        [str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}; see output above")

## 5. Dependencies

`torch` and `torchvision` are skipped: Colab ships builds matched to its CUDA driver, and
installing over them tends to fail later as a CUDA error inside training rather than here.

In [ ]:
PREINSTALLED = {"torch", "torchvision"}

wanted = []
for line in Path("requirements.txt").read_text().splitlines():
    line = line.split("#", 1)[0].strip()
    if not line:
        continue
    if line.split("==")[0].split(">=")[0].strip().lower() in PREINSTALLED:
        continue
    wanted.append(line)

# P2.7 only trains. ezkl, streamlit, onnx and cryptography belong to later
# phases and pulling them in here costs minutes of setup for nothing.
NEEDED_FOR_P2_7 = {"numpy", "matplotlib", "pytest"}
wanted = [w for w in wanted if w.split("==")[0].strip().lower() in NEEDED_FOR_P2_7]
print("installing:", wanted)
if wanted:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *wanted], check=True)
print("done")

## 6. Fast checks before spending GPU time

The unit tests, then every sweep run for 2 epochs on synthetic data with a public TEST-key bundle
(not your bundle). Both raise on failure, so Run all stops here. If either fails, report the
traceback. Do not force section 7.

In [ ]:
run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:warnings"])

In [ ]:
run([sys.executable, "experiments/p2_7_ratio_sweep.py", "--run", "all", "--smoke"])

## 7. The real runs, one cell each

Each cell trains one ratio. Before training it checks the bundle against
`EXPECTED_TRIGGER_SHA256` and against CIFAR-10 itself, and refuses a test-key bundle.

Every run gets its own Drive folders: checkpoints every epoch in
`{DRIVE_ROOT}/checkpoints/<run>/`, a result JSON in `{DRIVE_ROOT}/results/`, and once all 60 epochs
are done, the final weights in `{DRIVE_ROOT}/models/<run>_W_star.pt`.

**Re-running any cell is safe.** A run already complete on Drive, with its weights hash verified, is
skipped. A run that was cut off resumes from its last epoch. Do not change EPOCHS, SEED or
BATCH_SIZE between attempts: resuming under different settings is refused.

**What to watch.** `eval_acc` should climb as in P2.3; if epoch 2 still shows exactly `10.00%`,
stop and report it. `trigger_acc` is a training diagnostic.
- For the sparse runs (`r0001`, `r0005`) **it may stay low for the whole run.** That is a possible
  real result of the sweep, not a fault. Let the run finish.
- In `r0001` only 6 trigger samples go in per epoch, so the trigger set is seen about 3.6 times per
  trigger over the whole run.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then run all again.")


def sweep(run_name):
    run([
        sys.executable, "experiments/p2_7_ratio_sweep.py",
        "--run", run_name,
        "--drive-root", DRIVE_ROOT,
        "--data-root", DATA_ROOT,
        "--trigger-bundle", BUNDLE_PATH,
        "--expected-trigger-sha256", EXPECTED_TRIGGER_SHA256,
        "--epochs", EPOCHS,
        "--seed", SEED,
        "--batch-size", BATCH_SIZE,
        "--max-minutes", MAX_MINUTES,
        "--num-workers", 2,
    ])

In [ ]:
sweep("p2.7_r0001")

In [ ]:
sweep("p2.7_r0005")

In [ ]:
sweep("p2.7_r0020")

In [ ]:
sweep("p2.7_r0078")

In [ ]:
sweep("p2.7_r1252")

## 8. Collect what comes back

For each run, prints the newest result file. A run is final only when `stopped_early` is false
**and** its weights file is on Drive with a matching hash. Anything else means re-run that run's
cell in section 7.

In [ ]:
import hashlib
import json

RUNS = ["p2.7_r0001", "p2.7_r0005", "p2.7_r0020", "p2.7_r0078", "p2.7_r1252"]
incomplete = []
for name in RUNS:
    results = sorted(Path(DRIVE_ROOT, "results").glob(f"{name}__seed{SEED}__*.json"))
    weights = Path(DRIVE_ROOT, "models", f"{name}_W_star.pt")
    if not results:
        print(f"{name}: NO RESULT YET")
        incomplete.append(name)
        continue
    record = json.loads(results[-1].read_text())  # filenames end in a UTC timestamp
    m, p = record["metrics"], record["params"]
    ok_weights = weights.is_file() and hashlib.sha256(weights.read_bytes()).hexdigest() == m["weights_sha256"]
    if p["trigger_bundle"]["sha256"] != EXPECTED_TRIGGER_SHA256:
        raise RuntimeError(f"{name} was trained on a different bundle than expected")
    final = (not m["stopped_early"]) and ok_weights
    if not final:
        incomplete.append(name)
    print(f"{name}: {'COMPLETE' if final else 'INCOMPLETE'}")
    print(f"    result file        : {results[-1].name}")
    print(f"    epochs             : {m['epochs_completed']}/{m['epochs_planned']}  stopped_early={m['stopped_early']}")
    print(f"    tpb / every        : {p['triggers_per_batch']} / {p['trigger_every']}  ({p['trigger_samples_per_epoch']} trigger samples per epoch)")
    print(f"    test accuracy      : {m['test_accuracy'] * 100:.2f}%   holdout {m['holdout_accuracy'] * 100:.2f}%")
    print(f"    trigger accuracy   : {m['final_trigger_accuracy'] * 100:.1f}%  (training diagnostic, not the WDR)")
    print(f"    epoch time         : {m['total_epoch_seconds'] / 60:.1f} min, summed over sessions")
    print(f"    peak VRAM          : {m['peak_vram_mb_this_session']} MB (last session)   gpu {p['gpu']}")
    print(f"    code commit        : {record['git']['commit']} (dirty: {record['git']['dirty']})")
    print(f"    weights sha256     : {m['weights_sha256']}  file matches: {ok_weights}")

print()
if incomplete:
    print("*** NOT DONE:", ", ".join(incomplete), "-- re-run those cells in section 7. ***")
else:
    print("All five runs complete. Bring back the files listed in section 9.")

## 9. Bring back to the repo

For **each** of the five runs `p2.7_r0001`, `p2.7_r0005`, `p2.7_r0020`, `p2.7_r0078`, `p2.7_r1252`:

1. The **newest** `{DRIVE_ROOT}/results/<run>__seed1337__*.json` (the one section 8 printed) → the
   repo's `results/`. Older ones for the same run are incomplete sessions.
2. `{DRIVE_ROOT}/models/<run>_W_star.pt` (about 1.2 MB each) → the repo's `results/`, next to the
   JSON. `*.pt` is gitignored. Keep the Drive copies.

`checkpoints/<run>/best.pt` is **not** the model to bring back; it was picked by test accuracy.

Then report back. WDR and accuracy drop are measured locally from these files; nothing is recorded
before that. Finally run the cell below to flush Drive before the runtime goes away.

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
print("Drive flushed and unmounted. Remount (section 1) before running anything else.")